# Build EpistasisBench

Inputs live in `data/` (this notebook sits in `data/dataset_construction/`):

- `DMS_substitutions.csv` — ProteinGym v1.3 `reference_files/DMS_substitutions.csv`
  (https://github.com/OATML-Molecules/ProteinGym)
- `DMS_ProteinGym_substitutions/{DMS_id}.csv` — ProteinGym per-assay substitution files,
  the 50 Tsuboyama 2023 assays with multiple mutants
- `thermodynamic_coupling.csv` — output of `TC.py`, computed from the Tsuboyama 2023
  MegaScale Dataset2 + Dataset3 (https://zenodo.org/record/7992926)

Outputs: `../DMS_Tsuboyama.csv`, `../epi.csv`, `../EpistasisBench.csv`

In [1]:
import numpy as np
import pandas as pd
from pandas import read_csv, concat, isna

## Merge DMS data

In [2]:
tb = read_csv('../DMS_substitutions.csv')
tb = tb.loc[
    tb.includes_multiple_mutants & (tb.first_author == 'Tsuboyama')
]
tb.to_csv('../DMS_Tsuboyama.csv', index=False)

rt = []
for dms in tb.DMS_id:
    print('=', end='', flush=True)
    tb = read_csv(f'../DMS_ProteinGym_substitutions/{dms}.csv')[['mutant', 'DMS_score']]
    mut2ddG = dict(zip(tb.mutant, tb.DMS_score))
    dt = tb.loc[tb.mutant.str.contains(':')].copy().rename(columns={'DMS_score': 'ddG'})
    dt['dms'] = dms
    dt['mut1'], dt['mut2'] = zip(*dt.mutant.str.split(':'))
    dt['ddG1'] = dt.mut1.map(mut2ddG)
    dt['ddG2'] = dt.mut2.map(mut2ddG)
    n_multi = len(dt.index)
    dt = dt.dropna()
    rt.append(dt)

rt = concat(rt)
rt.to_csv('../epi.csv', index=False)
rt

=

=

=

=

=

=

... (41 outputs left out)

=

=

=

,mutant,ddG,dms,mut1,mut2,ddG1,ddG2
34,D28A:N41A,-0.868070,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41A,-1.218720,-1.554322
35,D28A:N41C,-0.109771,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41C,-1.218720,-0.902571
36,D28A:N41D,-1.004640,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41D,-1.218720,-2.439974
37,D28A:N41E,-1.255310,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41E,-1.218720,-2.967866
38,D28A:N41F,-0.497931,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41F,-1.218720,-1.722929
...,...,...,...,...,...,...,...
2258,T23Y:E25S,-1.962082,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25S,-2.562483,-0.308885
2259,T23Y:E25T,-2.233774,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25T,-2.562483,-0.423628
2260,T23Y:E25V,-2.120172,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25V,-2.562483,-0.495808
2261,T23Y:E25W,-1.749488,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25W,-2.562483,-0.434254


## ddG_int

In [3]:
tb = read_csv('../epi.csv')
tb['ddGint'] = tb.ddG - tb.ddG1 - tb.ddG2
tb

,mutant,ddG,dms,mut1,mut2,ddG1,ddG2,ddGint
0,D28A:N41A,-0.868070,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41A,-1.218720,-1.554322,1.904972
1,D28A:N41C,-0.109771,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41C,-1.218720,-0.902571,2.011519
2,D28A:N41D,-1.004640,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41D,-1.218720,-2.439974,2.654054
3,D28A:N41E,-1.255310,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41E,-1.218720,-2.967866,2.931276
4,D28A:N41F,-0.497931,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41F,-1.218720,-1.722929,2.443718
...,...,...,...,...,...,...,...,...
50807,T23Y:E25S,-1.962082,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25S,-2.562483,-0.308885,0.909286
50808,T23Y:E25T,-2.233774,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25T,-2.562483,-0.423628,0.752337
50809,T23Y:E25V,-2.120172,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25V,-2.562483,-0.495808,0.938118
50810,T23Y:E25W,-1.749488,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25W,-2.562483,-0.434254,1.247249


## Sign epistasis

In [4]:
eps = 0.07
cond1 = (tb.ddG1 + eps < 0) & (tb.ddG2 + eps < 0) & (tb.ddG - eps > np.maximum(tb.ddG1, tb.ddG2) + eps)
cond2 = (tb.ddG1 - eps > 0) & (tb.ddG2 - eps > 0) & (tb.ddG + eps < np.minimum(tb.ddG1, tb.ddG2) - eps)
cond3 = (tb.ddG1 + eps < 0) & (tb.ddG2 - eps > 0) & ((tb.ddG + eps < tb.ddG1 - eps) | (tb.ddG - eps > tb.ddG2 + eps))
cond4 = (tb.ddG1 - eps > 0) & (tb.ddG2 + eps < 0) & ((tb.ddG - eps > tb.ddG1 + eps) | (tb.ddG + eps < tb.ddG2 - eps))
tb['sign'] = cond1 | cond2 | cond3 | cond4
tb

,mutant,ddG,dms,mut1,mut2,ddG1,ddG2,ddGint,sign
0,D28A:N41A,-0.868070,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41A,-1.218720,-1.554322,1.904972,True
1,D28A:N41C,-0.109771,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41C,-1.218720,-0.902571,2.011519,True
2,D28A:N41D,-1.004640,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41D,-1.218720,-2.439974,2.654054,True
3,D28A:N41E,-1.255310,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41E,-1.218720,-2.967866,2.931276,False
4,D28A:N41F,-0.497931,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41F,-1.218720,-1.722929,2.443718,True
...,...,...,...,...,...,...,...,...,...
50807,T23Y:E25S,-1.962082,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25S,-2.562483,-0.308885,0.909286,False
50808,T23Y:E25T,-2.233774,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25T,-2.562483,-0.423628,0.752337,False
50809,T23Y:E25V,-2.120172,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25V,-2.562483,-0.495808,0.938118,False
50810,T23Y:E25W,-1.749488,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25W,-2.562483,-0.434254,1.247249,False


In [5]:
tb.to_csv('../epi.csv', index=False)

## Thermodynamic coupling

`TC.py` requires jax + numpyro and is slow. The precomputed `../thermodynamic_coupling.csv` is used below.

In [6]:
# !python TC.py

In [7]:
tb = read_csv('../thermodynamic_coupling.csv', low_memory=False)
print(tb.thermodynamic_coupling.map(isna).sum())
tb = tb[['name', 'pair_name', 'mut_type', 'WT_name', 'thermodynamic_coupling', 'ddG_ML']]
tb = tb.loc[tb.ddG_ML != '-']
tb['ddG_ML'] = tb.ddG_ML.astype(float)
print((~tb.mut_type.str.contains(':')).sum())
print(tb.WT_name.map(isna).sum())
print((tb.WT_name.str.split('.').str[0] != tb.name.str.split('.').str[0]).sum())
tb['pdb'] = tb.WT_name.str.split('.').str[0]
print((tb.pdb.map(len) != 4).sum())
tb = tb.loc[tb.pdb.map(len) == 4].copy()
tb = tb.rename(columns={'mut_type': 'mutant'})
tb = tb[['pdb', 'mutant', 'thermodynamic_coupling', 'ddG_ML']]

thermodynamic_coupling_dict = dict()
ddG_ML_dict = dict()
for pdb, mutant, thermodynamic_coupling, ddG_ML in tb.values:
    key = f'{pdb}|{mutant}'
    if key in ddG_ML_dict:
        ddG_ML_dict[key].append(ddG_ML)
        thermodynamic_coupling_dict[key].append(thermodynamic_coupling)
    else:
        ddG_ML_dict[key] = [ddG_ML]
        thermodynamic_coupling_dict[key] = [thermodynamic_coupling]

for key in ddG_ML_dict:
    ddG_ML_dict[key] = np.mean(ddG_ML_dict[key])
    thermodynamic_coupling_dict[key] = np.mean(thermodynamic_coupling_dict[key])

tb

0
0


0


0
4920


,pdb,mutant,thermodynamic_coupling,ddG_ML
0,1UFM,Q49Q:E53Q,-0.007675,-0.371867
1,1UFM,Q49Q:E53E,0.165806,0.031597
2,1UFM,Q49Q:E53N,0.033688,-0.688908
3,1UFM,Q49Q:E53H,-0.044883,-0.980420
4,1UFM,Q49Q:E53D,0.318886,-0.212474
...,...,...,...,...
210103,1W4G,L23F:G44M,-0.052258,-0.027629
210104,1W4G,L23F:G44C,0.206977,-0.099744
210105,1W4G,L23F:G44V,0.057392,-0.138505
210106,1W4G,L23F:G44W,0.084030,0.129047


In [8]:
dt = read_csv('../epi.csv')
dt['pdb'] = dt.dms.str.split('_').str[-1]
print((dt.pdb.map(len) != 4).sum())
dt['ddG_ML'] = (dt.pdb + '|' + dt.mutant).map(ddG_ML_dict)
dt['thermodynamic_coupling'] = (dt.pdb + '|' + dt.mutant).map(thermodynamic_coupling_dict)
print(((dt.ddG - dt.ddG_ML).abs() > 0.001).sum())
print(dt.thermodynamic_coupling.map(isna).sum())
dt.drop(columns=['pdb', 'ddG_ML']).to_csv('../epi.csv', index=False)
dt

0


0
0


,mutant,ddG,dms,mut1,mut2,ddG1,ddG2,ddGint,sign,pdb,ddG_ML,thermodynamic_coupling
0,D28A:N41A,-0.868070,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41A,-1.218720,-1.554322,1.904972,True,4G3O,-0.868070,-0.092582
1,D28A:N41C,-0.109771,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41C,-1.218720,-0.902571,2.011519,True,4G3O,-0.109771,-0.038285
2,D28A:N41D,-1.004640,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41D,-1.218720,-2.439974,2.654054,True,4G3O,-1.004640,0.123159
3,D28A:N41E,-1.255310,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41E,-1.218720,-2.967866,2.931276,False,4G3O,-1.255310,0.212276
4,D28A:N41F,-0.497931,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41F,-1.218720,-1.722929,2.443718,True,4G3O,-0.497931,-0.250613
...,...,...,...,...,...,...,...,...,...,...,...,...
50807,T23Y:E25S,-1.962082,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25S,-2.562483,-0.308885,0.909286,False,2JVD,-1.962082,0.016649
50808,T23Y:E25T,-2.233774,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25T,-2.562483,-0.423628,0.752337,False,2JVD,-2.233774,0.010218
50809,T23Y:E25V,-2.120172,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25V,-2.562483,-0.495808,0.938118,False,2JVD,-2.120172,0.079358
50810,T23Y:E25W,-1.749488,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25W,-2.562483,-0.434254,1.247249,False,2JVD,-1.749488,-0.014352


## Subsets

In [9]:
tb = read_csv('../epi.csv')
print((tb.loc[tb.sign].dms.value_counts() >= 30).sum())
print((tb.loc[tb.thermodynamic_coupling.abs() > 0.5].dms.value_counts() >= 30).sum())

tb

31
35


,mutant,ddG,dms,mut1,mut2,ddG1,ddG2,ddGint,sign,thermodynamic_coupling
0,D28A:N41A,-0.868070,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41A,-1.218720,-1.554322,1.904972,True,-0.092582
1,D28A:N41C,-0.109771,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41C,-1.218720,-0.902571,2.011519,True,-0.038285
2,D28A:N41D,-1.004640,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41D,-1.218720,-2.439974,2.654054,True,0.123159
3,D28A:N41E,-1.255310,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41E,-1.218720,-2.967866,2.931276,False,0.212276
4,D28A:N41F,-0.497931,AMFR_HUMAN_Tsuboyama_2023_4G3O,D28A,N41F,-1.218720,-1.722929,2.443718,True,-0.250613
...,...,...,...,...,...,...,...,...,...,...
50807,T23Y:E25S,-1.962082,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25S,-2.562483,-0.308885,0.909286,False,0.016649
50808,T23Y:E25T,-2.233774,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25T,-2.562483,-0.423628,0.752337,False,0.010218
50809,T23Y:E25V,-2.120172,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25V,-2.562483,-0.495808,0.938118,False,0.079358
50810,T23Y:E25W,-1.749488,YNZC_BACSU_Tsuboyama_2023_2JVD,T23Y,E25W,-2.562483,-0.434254,1.247249,False,-0.014352


In [10]:
tb = read_csv('../epi.csv')
tb['pos'] = tb.dms + '|' + tb.mut1.map(lambda x: x[1:-1]) + ':' + tb.mut2.map(lambda x: x[1:-1]) 
len(set(tb.pos))

180

In [11]:
sign = tb.loc[tb.sign]
print(sum(sign.dms.value_counts() >= 30))
tp = (sign.dms.value_counts() >= 30).reset_index()
sign = sign.loc[sign.dms.isin(tp.loc[tp['count']].dms)]
len(sign.mutant)

31


3267

In [12]:
strong = tb.loc[tb.thermodynamic_coupling.abs() > 0.5]
print(sum(strong.dms.value_counts() >= 30))
tp = (strong.dms.value_counts() >= 30).reset_index()
strong = strong.loc[strong.dms.isin(tp.loc[tp['count']].dms)]
len(strong.mutant)

35


6003

## Export

In [13]:
tb = read_csv('../epi.csv')
tb['uniprot_id'] = tb.dms.map(lambda x: x.split('_Tsuboyama_2023_')[0])
tb['pdb_id'] = tb.dms.map(lambda x: x.split('_Tsuboyama_2023_')[1])
tb["is_significant_epistasis"] = tb.thermodynamic_coupling.abs() > 0.5
tb = tb.rename(columns={
    "mutant": "double_mutant",
    "ddG": "ddG_AB",
    "mut1": "mutation_A",
    "mut2": "mutation_B",
    "ddG1": "ddG_A",
    "ddG2": "ddG_B",
    "ddGint": "ddG_int",
    "sign": "is_sign_epistasis",
    "thermodynamic_coupling": "thermodynamic_coupling_value"
})
tb[[
    'uniprot_id', 'pdb_id', 
    'double_mutant', 'mutation_A', 'mutation_B',
    'ddG_AB', 'ddG_A', 'ddG_B', 'ddG_int', 
    'thermodynamic_coupling_value',
    'is_sign_epistasis', 'is_significant_epistasis'
]].to_csv('../EpistasisBench.csv', index=False)